In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from tqdm import tqdm
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'biodiversity-risk').mkdir(parents=True, exist_ok=True)

# 6 — Biodiversity risk assessment (per CE scenario × β)

Overlays each projected mining footprint (`mining-area-expansion-2050-…`, from `3-mining-area-simulation/6-estimate-mining-area`) with rarity-weighted richness (RWR) to estimate the **mining-biodiversity overlap**: the fraction of each cell that is mined times the rarity-weighted richness it holds, summed globally and broken down by taxon and IUCN Red-List category.

This is a **risk / exposure** metric (potential range loss from projected expansion), not an observed impact.

## Cross (M/M central case)
scenario × β sweep — read footprints from `mining-area/`, write overlap outputs to `biodiversity-risk/`.

$$\text{mining-biodiversity overlap} = \sum_{\text{cells}} \frac{\text{footprint (km}^2)}{\text{cell area}} \times \text{RWR}$$

## Inputs
- footprint: `mining-area/mining-area-expansion-2050-{scenario}-B{beta}-{res}-D{Ld}-R{Lr}-A{La}.tif` (carries all three uncertainty dimensions; overlap output mirrors the same tag)
- RWR (overall + per taxon × IUCN category): `area-of-habitat-{res}/rarity-weighted-richness-*-{res}.tif`

In [3]:
def assess_biodiversity_risk(scenario, beta, demand_level='M', reserve_level='M', area_level='M', res='50km'):
    """mining-biodiversity overlap = (footprint / cell_area) * RWR, summed; broken down by
    taxon x IUCN category. A risk/exposure metric, not an observed impact.

    `tag` carries all three uncertainty dimensions (-DM-RM-AM central / -DL-RH-AL lower /
    -DH-RL-AH upper), matching the footprint filename written by 6-estimate-mining-area."""
    scale = {'10km': 10, '50km': 50}[res]
    tag = f'{scenario}-B{beta:.2f}-{res}-D{demand_level}-R{reserve_level}-A{area_level}'
    AOH = RESULTS / f'area-of-habitat-{res}'

    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    footprint = rio.open(RESULTS / 'mining-area' / f'mining-area-expansion-2050-{tag}.tif').read(1)
    if not np.isfinite(footprint).all() or (footprint < 0).any():
        raise ValueError(f'Invalid footprint for {tag}; regenerate it with 6-estimate-mining-area before assessing risk')
    RWR = rio.open(AOH / f'rarity-weighted-richness-vertebrates-{res}.tif').read(1)
    overlap = (footprint / scale ** 2) * RWR

    taxa = ['mammals', 'birds', 'amphibians', 'reptiles', 'vertebrates']
    df = pd.DataFrame(index=['CR', 'EN', 'VU', 'NT', 'LC/DD', 'total'], columns=taxa, dtype=float)
    for taxon in taxa:
        for category in ['CR', 'EN', 'VU', 'NT', 'total']:
            suffix = '' if category == 'total' else f'-{category}'
            RWR_t = rio.open(AOH / f'rarity-weighted-richness-{taxon}{suffix}-{res}.tif').read(1)
            df.loc[category, taxon] = float(np.sum((footprint / scale ** 2) * RWR_t))
    df.loc['LC/DD'] = df.loc['total'] - df.loc[['CR', 'EN', 'VU', 'NT']].sum()
    df = df.loc[['LC/DD', 'NT', 'VU', 'EN', 'CR'], ['mammals', 'birds', 'amphibians', 'reptiles']]

    out_tif = RESULTS / 'biodiversity-risk' / f'mining-biodiversity-overlap-2050-{tag}.tif'
    with rio.open(out_tif, 'w', driver='GTiff',
        height=country_raster_.shape[0], width=country_raster_.shape[1], count=1,
        dtype=overlap.dtype, crs=country_raster_.crs,
        transform=country_raster_.transform, compress='lzw') as dst:
        dst.write(overlap, 1)
    df.to_csv(RESULTS / 'biodiversity-risk' / f'mining-biodiversity-overlap-2050-{tag}.csv')

    print(f'  {tag}: mining-biodiversity overlap = {float(np.sum(overlap)):.4g}')
    return overlap, df

## Run — scenario × β cross (30, M/M)

In [ ]:
import json
_cfg = json.load(open(RESULTS / 'sweep-config.json'))   # sweep grid defined once in 5-2-map-extraction
SCENARIOS, BETAS = _cfg['SCENARIOS'], _cfg['BETAS']

records = []
for scen in tqdm(SCENARIOS):
    for b in BETAS:
        overlap, _ = assess_biodiversity_risk(scen, b, demand_level='M', reserve_level='M')
        records.append({'scenario': scen, 'beta': b, 'overlap': float(np.sum(overlap))})

risk_df = pd.DataFrame(records)
risk_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-summary-DM-RM.csv', index=False)
risk_df.round(3)

## Run — `scenario_set_1` × β0 × full 3×3×3 uncertainty factorial (216 overlaps)

Converts every footprint from the β0 demand×reserve×area factorial in `3-mining-area-simulation/6-estimate-mining-area` into mining-biodiversity overlap. Each call also writes the per-combo overlap raster + taxon×IUCN CSV; the tidy summary lands in `biodiversity-risk-summary-B0-DRA-factorial.csv` (216 rows).

→ `biodiversity-risk/mining-biodiversity-overlap-2050-{scen}-B0.00-50km-D{Ld}-R{Lr}-A{La}.tif` (+ taxon CSV)

In [4]:
scenario_set_1 = ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']
LEVELS = ['L', 'M', 'H']

# Convert every footprint from 6-estimate-mining-area's beta0 factorial into mining-biodiversity
# overlap. Each call also writes the per-combo overlap raster + taxon x IUCN CSV.
# 8 scenarios x {D,R,A} in {L,M,H}^3 = 216 overlaps, all at beta = 0.
records = []
for scen in tqdm(scenario_set_1):
    for dl in LEVELS:
        for rl in LEVELS:
            for al in LEVELS:
                overlap, _ = assess_biodiversity_risk(scen, 0.0, demand_level=dl,
                                                      reserve_level=rl, area_level=al)
                records.append({'scenario': scen, 'beta': 0.0, 'demand': dl,
                                'reserve': rl, 'area': al, 'overlap': float(np.sum(overlap))})

factorial_df = pd.DataFrame(records)
factorial_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-summary-B0-DRA-factorial.csv',
                    index=False)
factorial_df.round(3)

  0%|          | 0/8 [00:00<?, ?it/s]

  BAU-B0.00-50km-DL-RL-AL: mining-biodiversity overlap = 2.696
  BAU-B0.00-50km-DL-RL-AM: mining-biodiversity overlap = 3.109
  BAU-B0.00-50km-DL-RL-AH: mining-biodiversity overlap = 3.522
  BAU-B0.00-50km-DL-RM-AL: mining-biodiversity overlap = 2.071
  BAU-B0.00-50km-DL-RM-AM: mining-biodiversity overlap = 2.363
  BAU-B0.00-50km-DL-RM-AH: mining-biodiversity overlap = 2.655
  BAU-B0.00-50km-DL-RH-AL: mining-biodiversity overlap = 1.933
  BAU-B0.00-50km-DL-RH-AM: mining-biodiversity overlap = 2.207
  BAU-B0.00-50km-DL-RH-AH: mining-biodiversity overlap = 2.482
  BAU-B0.00-50km-DM-RL-AL: mining-biodiversity overlap = 3.152
  BAU-B0.00-50km-DM-RL-AM: mining-biodiversity overlap = 3.61
  BAU-B0.00-50km-DM-RL-AH: mining-biodiversity overlap = 4.069
  BAU-B0.00-50km-DM-RM-AL: mining-biodiversity overlap = 2.325
  BAU-B0.00-50km-DM-RM-AM: mining-biodiversity overlap = 2.637
  BAU-B0.00-50km-DM-RM-AH: mining-biodiversity overlap = 2.949
  BAU-B0.00-50km-DM-RH-AL: mining-biodiversity overlap =

 12%|█▎        | 1/8 [00:01<00:12,  1.73s/it]

  BAU-B0.00-50km-DH-RH-AL: mining-biodiversity overlap = 2.346
  BAU-B0.00-50km-DH-RH-AM: mining-biodiversity overlap = 2.653
  BAU-B0.00-50km-DH-RH-AH: mining-biodiversity overlap = 2.961
  MIU-B0.00-50km-DL-RL-AL: mining-biodiversity overlap = 2.485
  MIU-B0.00-50km-DL-RL-AM: mining-biodiversity overlap = 2.869
  MIU-B0.00-50km-DL-RL-AH: mining-biodiversity overlap = 3.252
  MIU-B0.00-50km-DL-RM-AL: mining-biodiversity overlap = 1.966
  MIU-B0.00-50km-DL-RM-AM: mining-biodiversity overlap = 2.249
  MIU-B0.00-50km-DL-RM-AH: mining-biodiversity overlap = 2.533
  MIU-B0.00-50km-DL-RH-AL: mining-biodiversity overlap = 1.843
  MIU-B0.00-50km-DL-RH-AM: mining-biodiversity overlap = 2.111
  MIU-B0.00-50km-DL-RH-AH: mining-biodiversity overlap = 2.379
  MIU-B0.00-50km-DM-RL-AL: mining-biodiversity overlap = 2.971
  MIU-B0.00-50km-DM-RL-AM: mining-biodiversity overlap = 3.413
  MIU-B0.00-50km-DM-RL-AH: mining-biodiversity overlap = 3.854
  MIU-B0.00-50km-DM-RM-AL: mining-biodiversity overlap 

 25%|██▌       | 2/8 [00:03<00:09,  1.65s/it]

...

 88%|████████▊ | 7/8 [00:10<00:01,  1.50s/it]

  CCR-B0.00-50km-DH-RM-AM: mining-biodiversity overlap = 2.836
  CCR-B0.00-50km-DH-RM-AH: mining-biodiversity overlap = 3.163
  CCR-B0.00-50km-DH-RH-AL: mining-biodiversity overlap = 2.287
  CCR-B0.00-50km-DH-RH-AM: mining-biodiversity overlap = 2.589
  CCR-B0.00-50km-DH-RH-AH: mining-biodiversity overlap = 2.891
  CE-B0.00-50km-DL-RL-AL: mining-biodiversity overlap = 2.181
  CE-B0.00-50km-DL-RL-AM: mining-biodiversity overlap = 2.526
  CE-B0.00-50km-DL-RL-AH: mining-biodiversity overlap = 2.87
  CE-B0.00-50km-DL-RM-AL: mining-biodiversity overlap = 1.8
  CE-B0.00-50km-DL-RM-AM: mining-biodiversity overlap = 2.07
  CE-B0.00-50km-DL-RM-AH: mining-biodiversity overlap = 2.34
  CE-B0.00-50km-DL-RH-AL: mining-biodiversity overlap = 1.699
  CE-B0.00-50km-DL-RH-AM: mining-biodiversity overlap = 1.956
  CE-B0.00-50km-DL-RH-AH: mining-biodiversity overlap = 2.213
  CE-B0.00-50km-DM-RL-AL: mining-biodiversity overlap = 2.498
  CE-B0.00-50km-DM-RL-AM: mining-biodiversity overlap = 2.881
  CE-B0.

100%|██████████| 8/8 [00:12<00:00,  1.54s/it]

  CE-B0.00-50km-DH-RH-AL: mining-biodiversity overlap = 2.015
  CE-B0.00-50km-DH-RH-AM: mining-biodiversity overlap = 2.296
  CE-B0.00-50km-DH-RH-AH: mining-biodiversity overlap = 2.577


,scenario,beta,demand,reserve,area,overlap
0,BAU,0.0,L,L,L,2.696
1,BAU,0.0,L,L,M,3.109
2,BAU,0.0,L,L,H,3.522
3,BAU,0.0,L,M,L,2.071
4,BAU,0.0,L,M,M,2.363
...,...,...,...,...,...,...
211,CE,0.0,H,M,M,2.476
212,CE,0.0,H,M,H,2.776
213,CE,0.0,H,H,L,2.015
214,CE,0.0,H,H,M,2.296


## Uncertainty bounds — anti-diagonal envelope at the 4 signposts

Overlays the lower (`DL-RH`) and upper (`DH-RL`) footprint corners with RWR. Writes per-combo overlap rasters + taxon×IUCN CSVs (for the figure error bars) and a tidy `biodiversity-risk-bounds-signposts.csv` (8 rows: lower/upper × BAU/CON/CE/IAP).

> **Why the anti-diagonal (not all-low/all-high).** The overlap is RWR-weighted, and the three sources do not move it together: demand and the area-CI band push it up, but **reserve pushes it down** (more reserve concentrates extraction into fewer, and lower-RWR-on-average, cells). A four-corner test confirmed the true extremes are `DL-RH` (low demand + high reserve) and `DH-RL` (high demand + low reserve); the old all-low/all-high diagonal does not bracket the central. This applies to both the footprint and the overlap.

In [ ]:
# --- Uncertainty bounds: overlay the anti-diagonal footprint corners with RWR to bracket the
#     mining-biodiversity overlap at the 4 signposts. Reserve runs opposite to demand, so the
#     lower/upper corners are DL-RH+areaL / DH-RL+areaH (NOT the all-low/all-high diagonal).
#     The central case (DM-RM-AM) is already in the sweep above.
#       -> mining-biodiversity-overlap-2050-{scen}-B{b}-50km-{DL-RH-AL|DH-RL-AH}.tif (+ taxon CSV)
SIGNPOSTS = [('BAU', 0.0), ('BAU', 0.6), ('CE', 0.0), ('CE', 0.6)]   # BAU / CON / CE / IAP
rows = []
for scen, b in SIGNPOSTS:
    for dl, rl, al, which in [('L', 'H', 'L', 'lower'), ('H', 'L', 'H', 'upper')]:
        overlap, _ = assess_biodiversity_risk(scen, b, demand_level=dl, reserve_level=rl, area_level=al)
        rows.append({'scenario': scen, 'beta': b, 'bound': which, 'overlap': float(np.sum(overlap))})

bounds_df = pd.DataFrame(rows)
bounds_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-bounds-signposts.csv', index=False)
bounds_df.round(4)